# Pipeline for Ca-Data-Preprocessing and Analysis

# Calcium Imaging preprocessing
---
* Authors: Yue Zhang, David Burkhardt, Tim Hladnik, Giulia Soto (AG Arrenberg)
* date: April 16, 2026


In [ ]:
# import libraries
from skimage import io
import os
import numpy as np
from PyQt5.QtWidgets import QFileDialog, QApplication

from Exp4_2026 import * 

# %matplotlib notebook
%matplotlib inline 
import matplotlib.pyplot as plt

In [ ]:
working_dir = "C:\\Users\\Arrenberg Lab\\Desktop\\preprocessing_arranberg_practical_2026-main\\test_data\\"
ca_filename = "test_data.TIF"
timeline_filename = "Io.hdf5"
stimulus_filename = "Display.hdf5"

In [ ]:
# load calcium data
ca_movie = load_ca_movie(working_dir+ca_filename)

## Registration

In [ ]:
# registration (motion correction) of calcium frames and computation of STD image
reg_frames, std_image = motion_correction(ca_movie, binsize=60, stepsize=40, crop_needed=True)    #10, 10

## Cell Segmentation

In [ ]:
# cell (ROIs) segmentation with watershed algorithm
segmentation_params = {
    'hpfiltSig': .1,
    'localThreKerSize': 13,
    'smoothSig': 3,
    'binaryThre': .5,
    'minSizeLim': 10,
    'maxSizeLim': 300,
    'bgKerSize': 2,
    'fgKerSize': 1
    }
roi_mask = cell_segmentation(std_image, segmentation_params, crop_needed=True)

In [ ]:
# extract calcium traces for ROIs
raw_ca_traces = extract_calcium_signals(roi_mask, reg_frames, display_traces=False)

# size of array
print(np.array(raw_ca_traces).shape)

## DF/F Computation
compute DF/f for each calcium trace
* Hint: $\frac{\Delta F}{F} = \frac{F(t) - F_{0}}{F_{0}}$

In [ ]:
dff_ca_traces = raw_ca_traces#None # put your calculation here

## Stimulus Parameter Aquisition

In [ ]:
# acquire stimulus parameter for each calcium frames
num_planes = 1
avg_per_layer = 15
stim_array = align_stimulus_to_ca_frames(timefn=working_dir+timeline_filename,stim_fn=working_dir+stimulus_filename, 
                                         num_planes=num_planes, avg_per_layer=avg_per_layer)

# visualize
fig, ax = plt.subplots(1, 2, figsize=(16, 8))
ax[0].set_title('Angular Velocities')
ax[0].plot(stim_array['time']-stim_array['time'][0],stim_array['ang_velocity'])
ax[0].plot(stim_array['time']-stim_array['time'][0],stim_array['phase'])
ax[0].set_xlabel('Time [s]')
ax[0].set_ylabel('angular velocity [°/s]')
ax[1].set_title('Angular Period')
ax[1].plot(stim_array['time']-stim_array['time'][0],stim_array['ang_period'])
ax[1].plot(stim_array['time']-stim_array['time'][0],stim_array['phase'])
ax[1].set_xlabel('Time [s]')
ax[1].set_ylabel('angular period [°/cyc]')

In [ ]:
# store ROIs and stimulus parameters  in one DataFrame
roi_id = np.unique(roi_mask)[1:]
formatted = pd.DataFrame(np.array(dff_ca_traces).T,columns=roi_id)
for k,v in stim_array.items():
    formatted["stim_"+k] = v
    
formatted = formatted[formatted['stim_phase'].notna()]


    
# visualize
# formatted.to_csv(working_dir + "test1.csv")
formatted

In [ ]:
# plot some example calcium traces
plt.figure()
for i in range(30):
    plt.plot(formatted['stim_time'],rnorm(formatted[i+1].to_numpy())+i)
plt.xlabel('#frames')
plt.ylabel('#ROIs')

## ROI Analysis

In [ ]:
# visualization of stimulus phase, velocity and a example ROI
fig, axes = plt.subplots(3, 1, figsize=(16, 10))
axes[0].plot(formatted['stim_phase'])
#axes[0].set_ylim(405, 485)
axes[0].set_ylabel('Phase #')

axes[1].plot(formatted['stim_ang_velocity'])
axes[1].set_ylabel('Velocity')

axes[2].plot(formatted[1])
axes[2].set_ylabel('Example ROI\'s DFF')

axes[2].set_xlabel('Frame #')

### Calculate mean DFF for stimulation phase

In [ ]:
# put your calculation here

### Calculate auto-correlation between stimulus repeats

In [ ]:
# put your calculation here

### Calculate correlation between mean phase-DFF and stimulus regressors

In [ ]:
# example plots for different regressors 
fig, axes = plt.subplots(3, 1, figsize=(16, 10))

axes[0].set_title('Example regressors')

axes[0].plot(formatted['stim_ang_velocity'] != 0)
# motion = formatted['stim_speed'] != 0
# reg_motion = CIRF(motion,n_ca_frames = 1059, rec_freq = 2)
# axes[0].plot(reg_motion)
# axes[0].set_xlim(0, 175)
axes[0].set_ylabel('Motion')

axes[1].plot(formatted['stim_ang_velocity'] > 0)
axes[1].set_ylabel('CW direction')

axes[2].plot(formatted['stim_ang_velocity'] < 0)
axes[2].set_ylabel('CCW direction')

axes[2].set_xlabel('Frame #')

In [ ]:
# put your calculation here